# Electricity Maps: Gold Layer RAG Chatbot

This notebook demonstrates how to create a Retrieval-Augmented Generation (RAG) system 
directly on top of our Gold Medallion Delta tables.

We will:
1. Read the Gold `daily_mix_summary` and `daily_imports` tables.
2. Convert structured rows into natural language documents.
3. Embed these documents into a local Vector DB (Chroma).
4. Use an LLM to answer questions by retrieving relevant data.

In [ ]:
import os
import polars as pl
from pathlib import Path
from dotenv import load_dotenv

# We use LangChain for the RAG orchestration
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Load environment variables (ensure LLM_API_KEY is set in config/env_prod.properties)
load_dotenv("../config/env_prod.properties")
llm_api_key = os.getenv("LLM_API_KEY")

if not llm_api_key or llm_api_key.startswith("your-") or llm_api_key == "":
    print("WARNING: Please set a valid OPENAI_API_KEY or LLM_API_KEY in env_prod.properties to run the LLM.")
else:
    # Langchain expects OPENAI_API_KEY by default
    os.environ["OPENAI_API_KEY"] = llm_api_key

# Set data dir
DATA_DIR = Path("../data")

### 1. Read Gold Tables
Let's read the daily summaries we created in the Gold layer.

In [ ]:
try:
    gold_mix = pl.read_delta(str(DATA_DIR / "gold" / "daily_mix_summary"))
    gold_imports = pl.read_delta(str(DATA_DIR / "gold" / "daily_imports"))
    
    print("Gold Mix Records:", len(gold_mix))
    print("Gold Imports Records:", len(gold_imports))
except Exception as e:
    print("Ensure you have run the pipeline first to generate the Gold tables!")
    raise e

### 2. Data to Text Conversion (Document Generation)
RAG systems (and LLMs) work best with natural language. We'll convert our structured rows into sentences.

In [ ]:
documents = []

# Process Mix Data
for row in gold_mix.iter_rows(named=True):
    date = row["date"]
    zone_name = row["zone_name"]
    source = row["source"]
    gen_mwh = row["generation_mwh"]
    pct = row["percentage"]
    
    text = f"On {date}, {zone_name} generated {gen_mwh:,.0f} MWh of {source} electricity, which accounted for {pct:.1f}% of its total energy mix."
    
    doc = Document(
        page_content=text,
        metadata={"date": str(date), "zone": zone_name, "type": "generation", "source": source}
    )
    documents.append(doc)

# Process Import Data
for row in gold_imports.iter_rows(named=True):
    date = row["date"]
    zone_name = row["zone_name"]
    source_zone = row["source_zone"]
    mwh = row["import_mwh"]
    
    text = f"On {date}, {zone_name} imported {mwh:,.0f} MWh of electricity from zone {source_zone}."
    
    doc = Document(
        page_content=text,
        metadata={"date": str(date), "zone": zone_name, "type": "import", "source_zone": source_zone}
    )
    documents.append(doc)

print(f"Generated {len(documents)} natural language documents from Gold data.")
print("Example:", documents[0].page_content)

### 3. Vector Database
We embed the documents and store them in ChromaDB.

In [ ]:
if not llm_api_key or llm_api_key.startswith("your-") or llm_api_key == "":
    print("Skipping Vector DB creation because no LLM API key is present.")
else:
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    
    # Create an ephemeral in-memory Chroma database
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embeddings,
        collection_name="electricity_maps_gold"
    )
    
    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    print("Vector database created and retriever initialized.")

### 4. Create the RAG Chain
We prompt the LLM to only use the retrieved context to answer.

In [ ]:
if not llm_api_key or llm_api_key.startswith("your-") or llm_api_key == "":
    print("Skipping Chain creation because no LLM API key is present.")
else:
    llm = ChatOpenAI(model=os.getenv("LLM_MODEL", "gpt-4o-mini"), temperature=0)

    template = """You are an energy data analyst for Electricity Maps.
    Answer the question based ONLY on the following context. 
    If you don't know the answer based on the context, say "I don't have enough data to answer that."

    Context:
    {context}

    Question: {question}
    """
    prompt = ChatPromptTemplate.from_template(template)

    def format_docs(docs):
        return "\n".join(doc.page_content for doc in docs)

    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    print("RAG chain ready!")

### 5. Test the Chatbot!

In [ ]:
def ask_chatbot(question: str):
    if not llm_api_key or llm_api_key.startswith("your-") or llm_api_key == "":
        return "LLM API key missing. Please check env_prod.properties."
    
    print(f"Question: {question}")
    answer = rag_chain.invoke(question)
    print(f"Answer: {answer}\n")

# Example Questions:
ask_chatbot("What percentage of France's electricity was nuclear yesterday?")
ask_chatbot("Did France import electricity from Spain (ES)? If so, how much?")
ask_chatbot("What was the lowest contributor to France's energy mix?")